In [189]:
#Importing necessary packages
from geopy import distance
import pandas as pd
import json
import ast
import numpy as np
import xarray as xr
import math
from datetime import datetime

# Step 1: Determine which buses are within 100 miles of the DAC

In [190]:
#Loading latitude and longitude coordinates for each bus
#UserInput
latLon_file_path = "/Users/lill771/Library/CloudStorage/OneDrive-PNNL/Documents/Results/GV_EGRET/Lats_Lons/LatLonSimp.csv"
latLon_df = pd.read_csv(latLon_file_path)
print(latLon_df.head())

   Bus_ID   Latitude   Longitude
0    2616  34.015353 -118.204014
1    2617  34.109480 -118.254629
2    3103  37.582985 -122.318546
3    3133  37.582985 -122.318546
4    3305  37.494231 -122.138182


### Information needed: Latitude and Longitude of DAC

In [191]:
#User input: latitude and longitude for the DAC of interest
DAC_lat = 35.98
DAC_lon = -119.24

In [192]:
#Calculating the distance (in miles) of each bus from the DAC using the geopy library
distances = []
for lat, lon in zip(latLon_df["Latitude"], latLon_df["Longitude"]):
    distances.append(distance.distance((lat,lon),(DAC_lat,DAC_lon)).miles)
latLon_df["Distance to DAC"] = distances
print(latLon_df.head())

   Bus_ID   Latitude   Longitude  Distance to DAC
0    2616  34.015353 -118.204014       147.630953
1    2617  34.109480 -118.254629       140.524180
2    3103  37.582985 -122.318546       203.393062
3    3133  37.582985 -122.318546       203.393062
4    3305  37.494231 -122.138182       191.749674


In [193]:
#Outputting a list of buses within 100 miles of the DAC
Tulare100_df = latLon_df[latLon_df["Distance to DAC"] <= 100]
bus_ids = Tulare100_df["Bus_ID"].tolist()

print("Bus IDs with Distance to DAC <= 100:")
print(bus_ids)

Bus IDs with Distance to DAC <= 100:
[3401, 3402, 3432, 3404, 3804, 3802, 3891, 3403, 3433, 3893, 3895, 3897, 3892, 3894, 3896, 3835, 3803, 3805, 3806, 3836]


In [207]:
print(latLon_df.loc[latLon_df["Bus_ID"]==3836])

     Bus_ID  Latitude   Longitude  Distance to DAC
214    3836  35.38543 -120.858669        99.848157


In [208]:
print(latLon_df.loc[latLon_df["Bus_ID"]==3433])

    Bus_ID   Latitude   Longitude  Distance to DAC
66    3433  36.609214 -119.639632        48.782597


In [209]:
print(latLon_df.loc[latLon_df["Bus_ID"]==3835])

     Bus_ID   Latitude   Longitude  Distance to DAC
135    3835  35.408481 -119.456003        41.232503


# Step 2: Generate dataframe with each generator, its fuel type, and its dispatch for every real time value

In [194]:
#Loading an example json file that has all of the generator, id, and fuel type information we need
json_file_path = '/Users/lill771/Library/CloudStorage/OneDrive-PNNL/Documents/GitHub/copper/run/python/osw_federates/da_energy_market_results_0.json'
with open(json_file_path,'r') as file:
    data = json.load(file)
print(data)

{'elements': {'bus': {'1001_FOURCORN_500': {'id': 1001, 'name': 'FOURCORN', 'base_kv': 500.0, 'matpower_bustype': 'PQ', 'vm': 1.0340100526809692, 'va': 5.578499794006348, 'area': 'SOUTH', 'zone': 3, 'owner': 1, 'v_min': 0.95, 'v_max': 1.05, 'lmp': {'data_type': 'time_series', 'values': [10057.26911094749, 9957.619426646248, 9957.619426646215, 9608.845501893955, 9608.845501893982, 9957.619426646215, 10804.64178775369, 10804.641787753675, 10804.641787753684, 9957.61942664622, 9185.334321340266, 9185.334321340266, 9185.334321340266, 9185.334321340266, 9185.334321340266, 9185.334321340251, 9185.334321340266, 10979.028750129775, 10804.641787753675, 10804.641787753675, 10804.641787753675, 10804.641787753675, 10804.641787753664, 10057.26911094749]}}, '1002_FOURCORN_345': {'id': 1002, 'name': 'FOURCORN', 'base_kv': 345.0, 'matpower_bustype': 'PQ', 'vm': 1.0140000581741333, 'va': 8.368499755859375, 'area': 'SOUTH', 'zone': 3, 'owner': 1, 'v_min': 0.95, 'v_max': 1.05, 'lmp': {'data_type': 'time_

In [195]:
#Generating a dataframe with each generator and its fuel type
generator_list = []
for generator_name, attributes in data["elements"]["generator"].items():
    fuel_type = attributes['fuel']
    generator_list.append({'Generator': generator_name, 'Fuel Type': fuel_type})
generator_df = pd.DataFrame(generator_list)
print(generator_df.head())

         Generator Fuel Type
0    COULEE_4131_B   Biomass
1  MIDPOINT_6132_B   Biomass
2  FCNGN4CC_1032_C      Coal
3  SJUAN G4_1034_C      Coal
4  CORONADO_1131_C      Coal


In [196]:
print(generator_df['Fuel Type'].unique())

['Biomass' 'Coal' 'Gas' 'Uranium' 'Wind' 'Solar' 'Hydro']


In [296]:
dispatch_file_path1 = '/Users/lill771/Downloads/MTDC_dispatch_1.csv'
dispatch_file_path2 = '/Users/lill771/Downloads/MTDC_dispatch_2.csv'
dispatch_df1 = pd.read_csv(dispatch_file_path1)
dispatch_df2 = pd.read_csv(dispatch_file_path2)
# Merge them on top of each other
dispatch_df = pd.concat([dispatch_df1, dispatch_df2], ignore_index=True)

print(dispatch_df)

                    real_time  sim_time              scenario federate  \
0      2032-01-01 00:00:00+00         0  osw_scenario_mtdc_v2  OSW_TSO   
1      2032-01-01 00:00:00+00         0  osw_scenario_mtdc_v2  OSW_TSO   
2      2032-01-01 00:00:00+00         0  osw_scenario_mtdc_v2  OSW_TSO   
3      2032-01-01 00:00:00+00         0  osw_scenario_mtdc_v2  OSW_TSO   
4      2032-01-01 00:00:00+00         0  osw_scenario_mtdc_v2  OSW_TSO   
...                       ...       ...                   ...      ...   
70582  2032-10-02 12:00:00+00  13867200  osw_scenario_mtdc_v4  OSW_TSO   
70583  2032-10-02 12:00:00+00  13867200  osw_scenario_mtdc_v4  OSW_TSO   
70584  2032-10-02 12:00:00+00  13867200  osw_scenario_mtdc_v4  OSW_TSO   
70585  2032-10-02 12:00:00+00  13867200  osw_scenario_mtdc_v4  OSW_TSO   
70586  2032-09-27 12:00:00+00  13435200  osw_scenario_mtdc_v4  OSW_TSO   

                                 data_name  \
0        OSW_TSO/da_dispatch_COULEE_4131_B   
1      OSW_TSO/da_d

In [297]:
#Generating a dataframe that contains all of the generator dispatch data and simplifying the Generator names
# dispatch_file_path = '/Users/lill771/Downloads/DaGeneratorDispatch.csv'
# dispatch_file_path = '/Users/lill771/Downloads/Radial_dispatch.csv'
# dispatch_df = pd.read_csv(dispatch_file_path)
dispatch_df['Generator'] = dispatch_df['data_name'].str.split('da_dispatch_').str[-1]

In [298]:
#Merging the generator and dispatch dataframes to include the generator fuel types
merged_df = pd.merge(dispatch_df, generator_df, on='Generator', how='left')
columns_to_keep = ['real_time', 'Generator', 'Fuel Type', 'data_value']
Dispatch_df = merged_df[columns_to_keep]

In [299]:
#Define a function to convert the string to a list of floats
#Doing this to convert the dispatch column to a list of floats
def convert_to_float_list(s):
    #Extract the substring starting from character position 39
    list_string = s[39:-1]
    #Evaluate the string to a Python list of floats using ast.literal_eval
    return ast.literal_eval(list_string)

In [300]:
#Converting data_value column to be a list of floats of dispatch values
Dispatch_df['Dispatch'] = Dispatch_df['data_value'].apply(convert_to_float_list)
Dispatch_df.drop('data_value', axis=1, inplace=True)
print(Dispatch_df.head())

                real_time        Generator Fuel Type  \
0  2032-01-01 00:00:00+00    COULEE_4131_B   Biomass   
1  2032-01-01 00:00:00+00  MIDPOINT_6132_B   Biomass   
2  2032-01-01 00:00:00+00  FCNGN4CC_1032_C      Coal   
3  2032-01-01 00:00:00+00  SJUAN G4_1034_C      Coal   
4  2032-01-01 00:00:00+00  CORONADO_1131_C      Coal   

                                            Dispatch  
0  [391.0500084757805, 391.0500084757805, 391.050...  
1  [122.0, 122.0, 67.10000145435333, 122.0, 122.0...  
2  [245.97180175781247, 245.97180175781247, 245.9...  
3  [547.7310180664062, 547.7310180664062, 547.731...  
4  [357.26486206054693, 357.26486206054693, 357.2...  


/var/folders/lb/mbncv83518x_zq8fw6p18lb40000gn/T/ipykernel_26608/1938614447.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Dispatch_df['Dispatch'] = Dispatch_df['data_value'].apply(convert_to_float_list)
/var/folders/lb/mbncv83518x_zq8fw6p18lb40000gn/T/ipykernel_26608/1938614447.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Dispatch_df.drop('data_value', axis=1, inplace=True)


# Step 3: Determine green house gas (GHG) emissions

In [301]:
#Emission rates in lbs CO2/MMBtu (source:https://www.nwcouncil.org/2021powerplan_greenhouse-gas-emissions-generation/)
emission_rates_mmbtu = {
    'Coal': 217.15, # average between 205.7 and 228.6 
    'Subbituminous': 214.3,
    'Bituminous': 205.7,
    'Petroleum/Oil': 161.3,
    'Gas': 117
}

#Conversion factor to lbs CO2/MWh: Multiply by 3.412 since 1 MWh = 3.412 MMBtu
emission_rates_mwh = {k: v * 3.412 for k, v in emission_rates_mmbtu.items()}

#Function to calculate emissions in lbs CO2
def calculate_emissions(row):
    fuel_type = row['Fuel Type']
    dispatch_values = row['Dispatch'] # MW
    emission_rate = emission_rates_mwh.get(fuel_type, 0) # lbs CO2/MWh
    return [value * emission_rate for value in dispatch_values]

In [302]:
#Apply the function to calculate the emissions and store results in a new column
Dispatch_df['Emissions (lbs CO2)'] = Dispatch_df.apply(calculate_emissions, axis=1)

#Factor to convert lbs CO2 to metric tons
conversion_factor = 0.000453592

#Apply the conversion to each list
Dispatch_df['Emissions (lbs CO2)'] = Dispatch_df['Emissions (lbs CO2)'].apply(lambda x: [value * conversion_factor for value in x])
Dispatch_df.rename(columns={"Emissions (lbs CO2)": "Emissions (metric tons CO2)"}, inplace=True)

/var/folders/lb/mbncv83518x_zq8fw6p18lb40000gn/T/ipykernel_26608/4224994502.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Dispatch_df['Emissions (lbs CO2)'] = Dispatch_df.apply(calculate_emissions, axis=1)
/var/folders/lb/mbncv83518x_zq8fw6p18lb40000gn/T/ipykernel_26608/4224994502.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Dispatch_df['Emissions (lbs CO2)'] = Dispatch_df['Emissions (lbs CO2)'].apply(lambda x: [value * conversion_factor for value in x])
/var/folders/lb/mbncv83518x_zq8fw6p18lb4

In [303]:
#Function to sum up all emissions
def sum_emissions(df):
    total_emission = sum(sum(emission_list) for emission_list in Dispatch_df['Emissions (metric tons CO2)'])
    return total_emission

#Calculate total emissions
total_emissions = sum_emissions(Dispatch_df)

print(f'Total Emissions (CO2e metric tons): {total_emissions}')

Total Emissions (CO2e metric tons): 73735811.56658234


# Step 4: Narrow down which buses within 100 miles of the DAC are contributing emissions

In [304]:
#Convert bus IDs within 100 miles to strings to be able to search the dataframe
Tulare_ids = [str(int(bus_id)) for bus_id in bus_ids]

In [305]:
#Filter out only generators within 100 miles and narrow down to fuel types that produce emissions: gas and coal
Tulare_df = Dispatch_df[Dispatch_df["Generator"].str.contains('|'.join(Tulare_ids))]
Tulare_df = Tulare_df[Tulare_df["Fuel Type"].isin(["Gas", "Coal"])]
unique_generators = Tulare_df["Generator"].unique()

#Print out buses that fit this description
print("Unique Generators:")
print(unique_generators)

Unique Generators:
['MORROBAY_3836_DG' 'MC CALL_3433_NG' 'MIDWAY_3835_NG']


# Step 5: Calculate emission rates in g/s of each pollutant: CO, NOx, SOx, PM2.5, and PM10

In [306]:
#Function that takes in generator dispatch data and outputs emission rate for CO, NOx, SOx, PM2.5, and PM10 in g/s
def calculate_emissions(row, pollutant):
    #Define constants for Coal (lb pollutant/ton coal) and Natural Gas (lb pollutant/10^6 scf)
    #Source: https://www.epa.gov/air-emissions-factors-and-quantification/ap-42-fifth-edition-volume-i-chapter-1-external-0
    pollutant_values = {
        "Coal": {"CO": 6, "NOx": 7.5, "SOx": 38, "PM2.5": 16.04, "PM10": 6},
        "Gas": {"CO": 84, "NOx": 190, "SOx": 0.6, "PM2.5": 7.6, "PM10": 7.6}
    }
    
    #Validate fuel type and pollutant
    if row["Fuel Type"] not in pollutant_values or pollutant not in pollutant_values[row["Fuel Type"]]:
        #Default: return a list of zeros based on dispatch length
        return [0] * len(row["Dispatch"])  

    #Get the fuel type and specific constant
    fuel_type = row["Fuel Type"]
    lbs_per_value = pollutant_values[fuel_type][pollutant]
    
    #Calculate lbs/MWh based on the fuel type
    if fuel_type == "Coal":
        #1.14 lbs coal/kWh (source: https://www.eia.gov/tools/faqs/faq.php?id=667&t=2)
        #(lbs pollutant/ton coal)(lbs coal/kWh)(1kWh/0.001MWh)(1ton coal/2000 lbs coal)=(lbs pollutant/MWh)
        lbs_per_mwh = lbs_per_value * 1.14 * (1 / 0.001) * (1 / 2000)  # Coal formula: lbs pollutant/MWh
    elif fuel_type == "Gas":
        #7.42 ft^3 gas/kWh (source: https://www.eia.gov/tools/faqs/faq.php?id=667&t=2)
        #(lbs pollutant/10^6 scf)(ft^3/kWh)(1kWh/0.001MWh)=(lbs pollutant/MWh)
        #Note that we are assuming 1scf=1ft^3
        lbs_per_mwh = lbs_per_value * (1/(10**6)) * 7.42 * (1 / 0.001) # Gas formula: lbs pollutant/MWh
    
    #Compute lbs pollutant/hr for each segment (multiplying by MWh)
    #The value if per hour since we have hourly data
    lbs_per_hr = [segment * lbs_per_mwh for segment in row["Dispatch"]]

    #Convert lbs pollutant/hr to grams pollutant/s
    #(453.592g/1lb)(1hr/3600s) 
    grams_per_s = [lbs * 453.592 / 3600 for lbs in lbs_per_hr]
    
    return grams_per_s

In [307]:
#Apply the calculate emissions function to calculate emission rates of each pollutant in g/s and add as new columns
Tulare_df["CO (g/s)"] = Tulare_df.apply(lambda row: calculate_emissions(row, "CO"), axis=1)
Tulare_df["NOx (g/s)"] = Tulare_df.apply(lambda row: calculate_emissions(row, "NOx"), axis=1)
Tulare_df["SOx (g/s)"] = Tulare_df.apply(lambda row: calculate_emissions(row, "SOx"), axis=1)
Tulare_df["PM2.5 (g/s)"] = Tulare_df.apply(lambda row: calculate_emissions(row, "PM2.5"), axis=1)
Tulare_df["PM10 (g/s)"] = Tulare_df.apply(lambda row: calculate_emissions(row, "PM10"), axis=1)

# Print the updated DataFrame
print("Updated DataFrame:")
print(Tulare_df.head())

Updated DataFrame:
                  real_time         Generator Fuel Type  \
23   2032-01-01 00:00:00+00  MORROBAY_3836_DG       Gas   
67   2032-01-01 00:00:00+00   MC CALL_3433_NG       Gas   
69   2032-01-01 00:00:00+00    MIDWAY_3835_NG       Gas   
276  2032-01-01 12:00:00+00  MORROBAY_3836_DG       Gas   
320  2032-01-01 12:00:00+00   MC CALL_3433_NG       Gas   

                                              Dispatch  \
23   [947.1234091653589, 835.0802935897655, 779.983...   
67   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
69   [1235.9609388858103, 1235.9609388858103, 1235....   
276  [612.8445628435948, 612.8445628435948, 779.983...   
320  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                           Emissions (metric tons CO2)  \
23   [171.50107291333092, 151.2127827625257, 141.23...   
67   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
69   [223.80254256906088, 223.80254256906088, 223.8...   
276  [110.97128319254506, 110.97128319254506,

# Step 6: Recording wind speed at generators of interest

In [212]:
#Load in relevant ERA5 wind data
file = '/Users/lill771/Documents/Data/ERA5/100m_uv_velocity.grib'
ds = xr.open_dataset(file, engine='cfgrib')  # Specify 'cfgrib' as the backend
ds = ds.to_dataframe().reset_index()
ds = ds[['time', 'latitude', 'longitude', 'u100', 'v100']]

/opt/anaconda3/envs/clean_pyenv/lib/python3.10/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [308]:
print(ds)

                         time  latitude  longitude      u100       v100
0         2016-01-01 00:00:00      55.0    -135.00 -0.798645  16.999069
1         2016-01-01 00:00:00      55.0    -134.75 -3.035950  19.137741
2         2016-01-01 00:00:00      55.0    -134.50 -5.168762  20.462936
3         2016-01-01 00:00:00      55.0    -134.25 -6.594543  19.245163
4         2016-01-01 00:00:00      55.0    -134.00 -8.128723  18.162155
...                       ...       ...        ...       ...        ...
748090723 2020-12-31 23:00:00      25.0    -101.00  8.238983   2.285065
748090724 2020-12-31 23:00:00      25.0    -100.75  8.036835   2.311432
748090725 2020-12-31 23:00:00      25.0    -100.50  8.166718   2.855377
748090726 2020-12-31 23:00:00      25.0    -100.25  6.792694   2.606354
748090727 2020-12-31 23:00:00      25.0    -100.00  1.560272   0.453033

[748090728 rows x 5 columns]


In [309]:
#User inputs (rounding lats and lons at each generator within 100 miles to the nearest 0.25)
Tulare_ERA5_lats = [35.5,36.5,35.5]
Tulare_ERA5_lons = [-120.75,-119.75,-119.5] 

In [310]:
#Storing lats and lons to corresponding bus ids
#User input to make list of ids here
Tulare_IDs = ['3836','3433','3835']
Tulare_tempWind_df = pd.DataFrame({
    'Bus ID': Tulare_IDs,
    'ERA5 lat': Tulare_ERA5_lats,
    'ERA5 lon': Tulare_ERA5_lons
})
print(Tulare_tempWind_df)

  Bus ID  ERA5 lat  ERA5 lon
0   3836      35.5   -120.75
1   3433      36.5   -119.75
2   3835      35.5   -119.50


In [311]:
#Making dataframe that has wind speeds at the generators of interest
TulareWindSpeeds = pd.DataFrame({})
for bus in Tulare_tempWind_df['Bus ID']:
    #% get lat/lon for each generator
    lat = list(Tulare_tempWind_df['ERA5 lat'].loc[Tulare_tempWind_df['Bus ID'] == bus])[0]
    long = list(Tulare_tempWind_df['ERA5 lon'].loc[Tulare_tempWind_df['Bus ID'] == bus])[0]
    
    #% find matching lat/lon for generator in ERA5 data
    temp_DF = ds.loc[(ds['latitude'] == lat) & (ds['longitude'] == long)]
    
    # Calculate windspeed from u and v values
    windspeed = (np.array(temp_DF['u100'])**2 + np.array(temp_DF['v100'])**2)**(1/2)
    
    # Save time and windspeed to dataframe
    if TulareWindSpeeds.empty:
        # Initialize with time and windspeed for the first bus
        TulareWindSpeeds = pd.DataFrame({
            'time': temp_DF['time'].values,  # Add time column explicitly
            bus: windspeed                  # Add windspeed for the first bus
        })
    else:
        # Add windspeed for subsequent buses
        TulareWindSpeeds[bus] = windspeed

# Ensure 'time' remains as a column
TulareWindSpeeds.reset_index(drop=True, inplace=True)

In [312]:
print(TulareWindSpeeds)

                     time      3836      3433      3835
0     2016-01-01 00:00:00  1.742052  2.953935  2.775891
1     2016-01-01 01:00:00  2.183205  4.232064  3.390436
2     2016-01-01 02:00:00  2.191647  4.802768  2.791743
3     2016-01-01 03:00:00  2.577947  4.659831  1.917454
4     2016-01-01 04:00:00  2.665110  4.672124  2.023814
...                   ...       ...       ...       ...
43843 2020-12-31 19:00:00  5.252923  7.141022  4.737538
43844 2020-12-31 20:00:00  5.355522  6.190084  3.651960
43845 2020-12-31 21:00:00  5.022228  4.598774  2.738015
43846 2020-12-31 22:00:00  3.692243  1.664634  1.477625
43847 2020-12-31 23:00:00  3.439767  2.909734  0.388193

[43848 rows x 4 columns]


In [313]:
#Reducing data to contain just one year's worth of data to match dispatch data
#Ensure 'time' column is in datetime format
TulareWindSpeeds['time'] = pd.to_datetime(TulareWindSpeeds['time'])

#Filter for rows where the year in 'time' is 2020
TulareWindSpeeds_2020 = TulareWindSpeeds[TulareWindSpeeds['time'].dt.year == 2020]

#Optional: Reset the index and keep the filtered dataframe clean
TulareWindSpeeds_2020.reset_index(drop=True, inplace=True)

In [314]:
print(TulareWindSpeeds_2020)

                    time      3836      3433      3835
0    2020-01-01 00:00:00  5.015583  2.277890  4.502339
1    2020-01-01 01:00:00  5.541583  1.852790  4.762964
2    2020-01-01 02:00:00  5.661527  1.325047  2.735762
3    2020-01-01 03:00:00  3.782397  0.635709  0.644354
4    2020-01-01 04:00:00  3.327144  0.178427  1.364741
...                  ...       ...       ...       ...
8779 2020-12-31 19:00:00  5.252923  7.141022  4.737538
8780 2020-12-31 20:00:00  5.355522  6.190084  3.651960
8781 2020-12-31 21:00:00  5.022228  4.598774  2.738015
8782 2020-12-31 22:00:00  3.692243  1.664634  1.477625
8783 2020-12-31 23:00:00  3.439767  2.909734  0.388193

[8784 rows x 4 columns]


In [315]:
#Convert to datetime
TulareWindSpeeds_2020['date'] = TulareWindSpeeds_2020['time'].dt.date

/var/folders/lb/mbncv83518x_zq8fw6p18lb40000gn/T/ipykernel_26608/3489167982.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TulareWindSpeeds_2020['date'] = TulareWindSpeeds_2020['time'].dt.date


In [316]:
#Since using day ahead, we have a list of 24 values per day, so changing format of wind data to match that
#User input: bus IDs
aggregated_wind_df = TulareWindSpeeds_2020.groupby('date').agg({
    '3836': list,
    '3433': list,
    '3835': list,
}).reset_index()

print(aggregated_wind_df)

           date                                               3836  \
0    2020-01-01  [5.01558256149292, 5.541583061218262, 5.661526...   
1    2020-01-02  [2.9746997356414795, 3.8496687412261963, 4.278...   
2    2020-01-03  [4.216857433319092, 4.136716365814209, 4.13714...   
3    2020-01-04  [3.2346861362457275, 3.359543800354004, 3.5874...   
4    2020-01-05  [3.5515193939208984, 5.034794330596924, 4.6490...   
..          ...                                                ...   
361  2020-12-27  [5.931142330169678, 6.145016670227051, 5.48064...   
362  2020-12-28  [3.80057954788208, 3.6835875511169434, 4.10693...   
363  2020-12-29  [3.271660327911377, 3.5369491577148438, 3.3365...   
364  2020-12-30  [4.573749542236328, 4.082901954650879, 3.91044...   
365  2020-12-31  [2.6334519386291504, 2.728477954864502, 2.2292...   

                                                  3433  \
0    [2.277890205383301, 1.8527898788452148, 1.3250...   
1    [2.799710750579834, 3.340684652328491,

In [317]:
#Create a new column in the dataframe for wind speeds
Tulare_df['wind speeds (m/s)'] = None

#Convert the real_time column to datetime
Tulare_df['real_time'] = pd.to_datetime(Tulare_df['real_time'])

In [318]:
#User input: generator to bus id map
generator_column_map = {
    "MORROBAY_3836_DG": "3836",
    "MC CALL_3433_NG": "3433",
    "MIDWAY_3835_NG": "3835",
}

In [319]:
#Iterate through each row and match the data
for index, row in Tulare_df.iterrows():
    #Get the relevant date and generator
    real_time_date = row['real_time']
    real_time_month_day = (real_time_date.month,real_time_date.day)
    generator = row['Generator']
    
    #Find the corresponding row in the wind data based on the date
    Tulare_wind_month_day = aggregated_wind_df['date'].apply(lambda d: (pd.to_datetime(d).month,pd.to_datetime(d).day))
    matching_row = aggregated_wind_df[Tulare_wind_month_day == real_time_month_day]
    
    if not matching_row.empty:
        #Get the values corresponding to the generator column
        generator_column = generator_column_map.get(generator, None)
        if generator_column:
            #Fetch the list of wind speeds directly (already stored as a list in the wind dataframe)
            wind_speeds_list = matching_row[generator_column].values[0]
            # Store the list in the DataFrame as-is
            Tulare_df.at[index, 'wind speeds (m/s)'] = wind_speeds_list

In [320]:
print(Tulare_df.head())

                    real_time         Generator Fuel Type  \
23  2032-01-01 00:00:00+00:00  MORROBAY_3836_DG       Gas   
67  2032-01-01 00:00:00+00:00   MC CALL_3433_NG       Gas   
69  2032-01-01 00:00:00+00:00    MIDWAY_3835_NG       Gas   
276 2032-01-01 12:00:00+00:00  MORROBAY_3836_DG       Gas   
320 2032-01-01 12:00:00+00:00   MC CALL_3433_NG       Gas   

                                              Dispatch  \
23   [947.1234091653589, 835.0802935897655, 779.983...   
67   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
69   [1235.9609388858103, 1235.9609388858103, 1235....   
276  [612.8445628435948, 612.8445628435948, 779.983...   
320  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                           Emissions (metric tons CO2)  \
23   [171.50107291333092, 151.2127827625257, 141.23...   
67   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
69   [223.80254256906088, 223.80254256906088, 223.8...   
276  [110.97128319254506, 110.97128319254506, 141.2.

# Step 7: Inserting Gaussian Dispersion Code

Complete equation for Gaussian dispersion modeling of continuous, buoyant air pollution plumes: $$C(x,y,z)=\frac{Q}{u}\cdot\frac{f}{\sigma_y\sqrt{2\pi}}\cdot\frac{g_1+g_2+g_3}{\sigma_z\sqrt{2\pi}},$$ where $C$ is the concentration of emissions in g/m$^3$ at any receptor located in position $(x,y,z)$. Source: https://en.wikipedia.org/wiki/Atmospheric_dispersion_modeling. In this equation, $x$ is the number of meters downwind from the emission source point, $y$ is the number of meters crosswind from the emission plume centerline, and $z$ is the number of meters above ground level.

In [321]:
#These are constants that are reasonably well known and can be assumed
g = 9.8 #acceleration due to gravity (m/s^2)
dTa_dz = -0.27965 #rate of temperature change with altitude (K/m) 
                #Source: https://scied.ucar.edu/learning-zone/atmosphere/change-atmosphere-altitude
Γ = 0.28295 #adiabatic lapse rate (K/m) Source:
        #https://forecast.weather.gov/glossary.php?word=lapse%20rate#:~:text=Adiabatic%20Lapse%20Rate,9.8%C2%B0C%20per%20kilometer

In [322]:
#reasonable values for all input quantities based on sources (can be changed once given actual data)
Hs = 91.44 #height of stack (m)
          #value is average stack height
          #Source: https://www.tandfonline.com/doi/pdf/10.1080/00022470.1963.10468165
w0 = 9.92 #initial plume vertical speed (m/s) [usually taken as stack exit velocity]
          #mean exit gas velocity as reported by the EPA in 1999
          #Source: https://www.epa.gov/sites/default/files/2014-03/documents/stacks99.pdf
D = 10 #internal stack diameter (m)
       #average diameter of industrial smoke stack
       #Source: https://www.nist.gov/news-events/news/2014/08/scale-model-smokestack-study-ghg-emissions-monitoring
Ts = 463.15 #stack gas temperature (K)
            #average value stated in EPA Stack Gas Reheat Evaluation in 1980 (converted from F to K)
            #Source: https://nepis.epa.gov/Exe/ZyNET.exe/...
#USER INPUT (temperature)!!
Ta = 290.628923889 #air temperature (K)
                # average ambient temperature in Tulare county, CA
                # source: epa.maps.arcgis.com/apps/webappviewer/indexhtml?id=...
L = 240 #height from ground level to bottom of ht einversion aloft (m) 
        #value is average in Allegheny County, Pennsylvania over a 10 year period; Source:
        #https://www.alleghenycounty.us/files/assets/county/v/1/government/health/documents/air-quality/sadar-emplus-article-reprint.pdf

In [323]:
#Location of desired concentration of emissions relative to the emission source
#User input
x1 = 160690.032379 #number of meters downwind from 3836
x2 = 78507.9797864 #number of meters downwind from 3433
x3 = 66357.281308 #number of meters downwind from 3835
y = 0 #number of meters crosswind from the emission plume centerline we are interested in calculating concentration at
z = Hs #number of meters above ground level we are interested in calculating concentration at

We now aim to calculate the horizontal and vertical standard deviations of the emission distrubution, $\sigma_y$ and $\sigma_z$, in m. The equations for these are given by:
$$\sigma_y=\exp(I_y+J_y\ln(x)+K_y[\ln(x)]^2),\quad \quad\sigma_z=\exp(I_z+J_z\ln(x)+K_z[\ln(x)]^2).$$
Note here that the coefficients, $I_y$, $I_z$, $J_y$, $J_z$, $K_y$, and $K_z$ are given by the values in the following table based on stability class:

![Alt Text](/Users/lill771/Downloads/StabilityClass2.png)

The classification of stability class is proposed by F. Pasquill. The six stability classes are referred to: A-extremely unstable B-moderately unstable C-slightly unstable D-neutral E-slightly stable F-moderately stable. One can determine the stability class based on surface wind speed and sky conditions using the following table. Source: https://dc.uwm.edu/cgi/viewcontent.cgi?article=2458&context=etd.

![Alt Text](/Users/lill771/Downloads/Pasquill.png)

The crosswind dispersion parameter is computed via: 
$$f=\exp\left(\frac{-y^2}{2\sigma_y^2}\right).$$

$H$ is the height of emission plume centerline above ground level in meters. It is made up of the actual stack height, $Hs$, and the plume rise, $\Delta h$. As such, we can compute $H$ via:
$$H = Hs + \Delta h.$$

While we are assuming that $Hs$ is given, determining $\Delta h$ requires some further thought and computation. We can use Briggs plume rise equations to determine $\Delta h$. A logic diagram for using the Briggs equations to obtain the plume rise trajectory of bent-over buoyant plumes is presented below:

![Alt Text](/Users/lill771/Downloads/Logic.png)

We see that the logic diagram depends on the buoyancy factor, $F$. This is given by:
$$F=w_0g\left(\frac{D}{2}\right)^2\left(\frac{T_s}{T_a}-1\right),$$
where $w_0$ is the initial plume vertical speed (m/s), $g$ is the acceleration due to gravity (m/s$^2$), $D$ is the internal stack diameter (m), $T_s$ is the stack gas temperature (K), and $T_a$ is the air temperature (K). Note that $w_0$ is usally taken as the stack exit velocity. We are going to take each of these quantities as given. Source: https://www.euromotor.org/mod/resource/view.php?id=14613&forceview=1. Note that there was a typo in the formula on the source website, but the typo is fixed here.

The logic diagram also depends on the stability paramter, $s$. This is given by: 
$$s=\left(\frac{g}{T_a}\right)\left[\left(\frac{\mathrm{d}T_a}{\mathrm{d}z}\right)+\Gamma\right],$$
where $\frac{\mathrm{d}T_a}{\mathrm{d}z}$ is the rate of temperature change with altitude and $\Gamma$ is the adiabatic lapse rate. Source: https://www.euromotor.org/mod/resource/view.php?id=14613&forceview=1. 

Computing the vertical dispersion parameter $g_1+g_2+g_3$: We begin with the vertical dispersion parameter with no reflections, $g1$, given by:
$$g_1=\exp\left[\frac{-(z-H)^2}{2\sigma_z^2}\right].$$

We now compute the vertical dispersion for reflection from the ground, $g_2$, given by:
$$g_2=\exp\left[\frac{-(z+H)^2}{2\sigma_z^2}\right].$$

Finally, we compute the vertical dispersion for reflection from an inversion aloft, $g_3$. Note that should we choose not to include the inversion aloft in the model, that this term will be omitted from the calculation of the concentration of emissions. The equation for $g_3$ is given by:
$$g_3=\sum_{m=1}^\infty\left\{\exp\left[\frac{-(z-H-2mL)^2}{2\sigma_z^2}\right]+\exp\left[\frac{-(z+H+2mL)^2}{2\sigma_z^2}\right]+\exp\left[\frac{-(z+H-2mL)^2}{2\sigma_z^2}\right]+\exp\left[\frac{-(z-H+2mL)^2}{2\sigma_z^2}\right]\right\}.$$
We note that according to the source with the model, truncating the sum to three terms generally gives an adequate solution, so that is what we do here.

In [324]:
def σ(u,daytime,x):
    #determine values of coefficients Iy, Jy, Ky, Iz, Jz, and Kz
    if u < 2:
        if daytime == True:
            #taking average between A and B values
            Iy = (-1.104 + -1.634) / 2
            Jy = (0.9878 + 1.0350) / 2
            Ky = (-0.0076 + -0.0096) / 2
            Iz = (4.679 + -1.999) / 2
            Jz = (-1.7172 + 0.8752) / 2
            Kz = (0.2770 + 0.0136) / 2
            stabEF = False
        else:
            #taking average between E and F values
            Iy = (-2.754 + -3.143) / 2
            Jy = (1.0106 + 1.0148) / 2
            Ky = (-0.0064 + -0.0070) / 2
            Iz = (-3.783 + -4.490) / 2
            Jz = (1.3010 + 1.4024) / 2
            Kz = (-0.0450 + -0.0540) /2
            stabEF = True
    elif (u >= 2) and (u < 3):
        if daytime == True:
            #taking "weighted average" over A, B, and C values
            Iy = ((0.5*(-1.104)) + (1.5*(-1.634)) + -2.0554) / 3
            Jy = ((0.5*0.9878) + (1.5*1.0350) + 1.0231) / 3
            Ky = ((0.5*(-0.0076)) + (1.5*(-0.0096)) + -0.0076) / 3
            Iz = ((0.5*4.679) + (1.5*(-1.999)) + -2.341) / 3
            Jz = ((0.5*(-1.7172)) + (1.5*0.8752) + 0.9477) / 3
            Kz = ((0.5*0.2770) + (1.5*0.0136) + -0.0020) / 3
            stabEF = False
        else:
            #taking average between E and F values
            Iy = (-2.754 + -3.143) / 2
            Jy = (1.0106 + 1.0148) / 2
            Ky = (-0.0064 + -0.0070) / 2
            Iz = (-3.783 + -4.490) / 2
            Jz = (1.3010 + 1.4024) / 2
            Kz = (-0.0450 + -0.0540) /2 
            stabEF = True
    elif (u >= 3) and (u < 5):
        if daytime == True:
            #taking average between B and C values
            Iy = (-1.634 + -2.054) / 2
            Jy = (1.0350 + 1.0231) / 2
            Ky = (-0.0096 + -0.0076) / 2
            Iz = (-1.999 + -2.341) / 2
            Jz = (0.8752 + 0.9477) / 2
            Kz = (0.0136 + -0.0020) / 2
            stabEF = False
        else:
            #taking average between D and E values
            Iy = (-2.555 + -2.754) / 2
            Jy = (1.0423 + 1.0106) / 2
            Ky = (-0.0087 + -0.0064) / 2
            Iz = (-3.186 + -3.783) / 2
            Jz = (1.1737 + 1.3010) / 2
            Kz = (-0.0316 + -0.0450) / 2
            stabEF = True #could technically also be False; making a choice here
    elif (u >= 5) and (u <= 6):
        if daytime == True:
            #taking average between C and D values
            Iy = (-2.054 + -2.555) / 2
            Jy = (1.0231 + 1.0423) / 2
            Ky = (-0.0076 + -0.0087) / 2
            Iz = (-2.341 + -3.186) / 2
            Jz = (0.9477 + 1.1737) / 2
            Kz = (-0.0020 + -0.0316) / 2
            stabEF = False
        else:
            #taking D values
            Iy = -2.555
            Jy = 1.0423
            Ky = -0.0087
            Iz = -3.186
            Jz = 1.1737
            Kz = -0.0316
            stabEF = False        
    else:
        if daytime == True:
            #taking "weighted average" of C and D values
            Iy = (-2.054 + (2*(-2.555))) / 3
            Jy = (1.0231 + (2*1.0423)) / 3
            Ky = (-0.0076 + (2*(-0.0087))) / 3
            Iz = (-2.341 + (2*(-3.186))) / 3
            Jz = (0.9477 + (2*1.1737)) / 3
            Kz = (-0.0020 + (2*(-0.0316))) / 3
            stabEF = False        
        else: 
            #taking D values
            Iy = -2.555
            Jy = 1.0423
            Ky = -0.0087
            Iz = -3.186
            Jz = 1.1737
            Kz = -0.0316
            stabEF = False 

    σy = math.exp(Iy+(Jy*math.log(x))+(Ky*((math.log(x))**2)))
    σz = math.exp(Iz+(Jz*math.log(x))+(Kz*((math.log(x))**2)))

    return σy,σz,stabEF

In [325]:
def deltaH(F,stabEF,u,s,x):
    #determine xf value
    if F >= 55:
        xf = 119*(F**0.40)
    else: 
        xf = 49*(F**0.625)
    
    #continue down logic diagram
    if stabEF == True:
        if (1.84*u*(s**(-0.5))) >= xf:
            if x < xf:
                dh = 1.6*(F**(1/3))*(x**(2/3))*(u**(-1))
            else:
                dh = 1.6*(F**(1/3))*(xf**(2/3))*(u**(-1))
        else:
            if x < (1.84*u*(s**(-0.5))):
                dh = 1.6*(F**(1/3))*(x**(2/3))*(u**(-1))
            else:
                dh = 2.4*((F/(u*s))**(1/3))
    else:
        if x < xf:
            dh = 1.6*(F**(1/3))*(x**(2/3))*(u**(-1))
        else:
            dh = 1.6*(F**(1/3))*(xf**(2/3))*(u**(-1))

    return dh

In [326]:
def g3Val(z,H,L,σz):
    g3 = 0.0
    for m in range(1,3+1):
        term1 = math.exp((-((z-H-(2*m*L))**2))/(2*(σz**2))) + math.exp((-((z+H+(2*m*L))**2))/(2*(σz**2)))
        term2 = math.exp((-((z+H-(2*m*L))**2))/(2*(σz**2))) + math.exp((-((z-H+(2*m*L))**2))/(2*(σz**2)))
        g3 += term1 + term2
    return g3

In [327]:
def Concentration(x,y,z,Q,u,time):
    daytime = time >= datetime.strptime("06:00:00", "%H:%M:%S").time() and time < datetime.strptime("18:00:00", "%H:%M:%S").time()
    σy,σz,stabEF = σ(u,daytime,x)
    f = math.exp((-y**2)/(2*(σy**2)))
    F = w0*g*((D/2)**2)*((Ts/Ta)-1)
    s = (g/Ta)*(dTa_dz + Γ)
    dh = deltaH(F,stabEF,u,s,x)
    H = Hs + dh
    g1 = math.exp((-((z-H)**2))/(2*(σz**2)))
    g2 = math.exp((-((z+H)**2))/(2*(σz**2)))
    g3 = g3Val(z,H,L,σz)
    C = (Q/u)*(f/(σy*math.sqrt(2*math.pi)))*((g1+g2+g3)/(σz*math.sqrt(2*math.pi)))
    return C

# Step 8: Applying dispersion code to emission rate data to get concentration of each pollutant in DAC

In [328]:
# List of pollutants (columns) requiring calculation
gas_columns = ["CO (g/s)", "NOx (g/s)", "SOx (g/s)", "PM2.5 (g/s)", "PM10 (g/s)"]

generator_x_map = {
    "MORROBAY_3836_DG": x1,
    "MC CALL_3433_NG": x2,
    "MIDWAY_3835_NG": x3,
}

# Process each row and update the DataFrame with new calculated columns of pollutant concentrations
for gas in gas_columns:
    # USER INPUT!!!
    Tulare_df[f"{gas}_result"] = Tulare_df.apply(
        lambda row: [
            # Determine which x-value to use based on the "Generator" condition
            Concentration(
                generator_x_map[row["Generator"]],
                y,
                z,
                concentration,
                wind_speed,
                row["real_time"].time()
            )
            for concentration, wind_speed in zip(row[gas], row["wind speeds (m/s)"])
        ],
        axis=1,
    )

In [329]:
#Check results - df now has new columns for each pollutant with concentration values
print(Tulare_df.head())  #Display the first few rows of the DataFrame

                    real_time         Generator Fuel Type  \
23  2032-01-01 00:00:00+00:00  MORROBAY_3836_DG       Gas   
67  2032-01-01 00:00:00+00:00   MC CALL_3433_NG       Gas   
69  2032-01-01 00:00:00+00:00    MIDWAY_3835_NG       Gas   
276 2032-01-01 12:00:00+00:00  MORROBAY_3836_DG       Gas   
320 2032-01-01 12:00:00+00:00   MC CALL_3433_NG       Gas   

                                              Dispatch  \
23   [947.1234091653589, 835.0802935897655, 779.983...   
67   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
69   [1235.9609388858103, 1235.9609388858103, 1235....   
276  [612.8445628435948, 612.8445628435948, 779.983...   
320  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                           Emissions (metric tons CO2)  \
23   [171.50107291333092, 151.2127827625257, 141.23...   
67   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
69   [223.80254256906088, 223.80254256906088, 223.8...   
276  [110.97128319254506, 110.97128319254506, 141.2.

In [330]:
#Add up the results from each generator to get total concentration values in the DAC
#Custom function for element-wise addition of lists
def sum_lists(series):
    return [sum(values) for values in zip(*series)]

#List of pollutant result columns
result_columns = [f"{gas}_result" for gas in gas_columns]

#Group by "real_time" and sum rows with element-wise addition for the result columns
Tulare_summed_df = Tulare_df.groupby("real_time", as_index=False).agg(
    {col: lambda x: sum_lists(x) for col in result_columns}
)

print(Tulare_summed_df.head())

                  real_time  \
0 2032-01-01 00:00:00+00:00   
1 2032-01-01 12:00:00+00:00   
2 2032-01-02 12:00:00+00:00   
3 2032-01-03 12:00:00+00:00   
4 2032-01-04 12:00:00+00:00   

                                     CO (g/s)_result  \
0  [1.8542279322883094e-05, 1.6964808966423335e-0...   
1  [1.2954760646187115e-06, 1.1823478272488636e-0...   
2  [9.54669367090777e-08, 6.734632018418635e-07, ...   
3  [1.657721168752294e-07, 1.6897459625223758e-07...   
4  [5.260600472007679e-07, 4.6606538085640416e-07...   

                                    NOx (g/s)_result  \
0  [4.194086989699748e-05, 3.837278218595754e-05,...   
1  [2.930243479494705e-06, 2.674358180681954e-06,...   
2  [2.1593711874672338e-07, 1.5233096232137388e-0...   
3  [3.7496074055111415e-07, 3.822044439038708e-07...   
4  [1.1898977258112606e-06, 1.054195504318057e-06...   

                                    SOx (g/s)_result  \
0  [1.3244485230630782e-07, 1.2117720690302382e-0...   
1  [9.253400461562226e-09, 

In [331]:
#Convert concentration values from g/m^3 to micograms/m^3
for col in result_columns:
    Tulare_summed_df[col] = Tulare_summed_df[col].apply(
        lambda lst: [val * 1e6 for val in lst] if isinstance(lst, list) else lst
    )

In [332]:
#Conversion factors
#Source: https://www.breeze-technologies.de/blog/air-pollution-how-to-convert-between-mgm3-%C2%B5gm3-ppm-ppb/
CO_conversion_ppm = 1.15 * 1000  # micrograms/m^3 to ppm
SO2_conversion_ppb = 2.62        # micrograms/m^3 to ppb
NO2_conversion_ppb = 1.88        # micrograms/m^3 to ppb

#Create "CO (ppm)" column
Tulare_summed_df["CO (ppm)"] = Tulare_summed_df["CO (g/s)_result"].apply(
    lambda lst: [value / CO_conversion_ppm for value in lst]
)

#Create "SO2 (ppb)" column
Tulare_summed_df["SO2 (ppb)"] = Tulare_summed_df["SOx (g/s)_result"].apply(
    lambda lst: [value / SO2_conversion_ppb for value in lst]
)

#Create "NO2 (ppb)" column
Tulare_summed_df["NO2 (ppb)"] = Tulare_summed_df["NOx (g/s)_result"].apply(
    lambda lst: [value / NO2_conversion_ppb for value in lst]
)

In [333]:
#Rename columns to have the appropriate units
Tulare_summed_df.rename(
    columns={
        "PM2.5 (g/s)_result": "PM2.5 (micrograms/m^3)",
        "PM10 (g/s)_result": "PM10 (micrograms/m^3)"
    },
    inplace=True  #Make changes directly to the DataFrame
)

# Verify successful renaming
print(Tulare_summed_df.columns)  #Print all column names to confirm

Index(['real_time', 'CO (g/s)_result', 'NOx (g/s)_result', 'SOx (g/s)_result',
       'PM2.5 (micrograms/m^3)', 'PM10 (micrograms/m^3)', 'CO (ppm)',
       'SO2 (ppb)', 'NO2 (ppb)'],
      dtype='object')


In [334]:
#Calculating the average concentration levels of each pollutant that is contributed from generator emission
columns_to_average = [
    "PM2.5 (micrograms/m^3)",
    "PM10 (micrograms/m^3)",
    "CO (ppm)",
    "SO2 (ppb)",
    "NO2 (ppb)"
]

# Dictionary to store the overall average for each column
column_averages = {}

# Compute the average for each column
for column in columns_to_average:
    # Flatten all lists in the column into a single list
    all_values = [value for row in Tulare_summed_df[column] for value in row]
    
    # Compute the average of all values in the column
    column_average = sum(all_values) / len(all_values)
    column_averages[column] = column_average

# Display the averages
print(column_averages)

{'PM2.5 (micrograms/m^3)': 0.1136674356457335, 'PM10 (micrograms/m^3)': 0.1136674356457335, 'CO (ppm)': 0.0010924559032313074, 'SO2 (ppb)': 0.0034250934806870506, 'NO2 (ppb)': 1.5115350484805108}


In [335]:
#MAJOR USER INPUT!!!! Need to determine background levels of each pollutant in the DAC of interest
#I used https://epa.maps.arcgis.com/apps/webappviewer/index.html?id=5f239fd3e72f424f98ef3d5def547eb5&extent=-146.2334,13.1913,-46.3896,56.5319 to
#find overall background values and then subtracted off the average concentration of each emission calculated above
# overall background levels
# background_levels = {
#     'PM2.5 (micrograms/m^3)': 9.746409,
#     'PM10 (micrograms/m^3)': 26.469613,
#     'CO (ppm)': 0.409951,
#     'SO2 (ppb)': 1.378092,
#     'NO2 (ppb)': 15.879259,
# }
# background levels with average pollutant concentration from emissions subtracted off
background_levels = {
    'PM2.5 (micrograms/m^3)': 9.59008591054,
    'PM10 (micrograms/m^3)': 26.3132899105,
    'CO (ppm)': 0.40844858129,
    'SO2 (ppb)': 1.37338158147,
    'NO2 (ppb)': 13.8004945125,
}

# Step 9: Inserting AQI code

This calculation is based on the steps given in: https://www.airnow.gov/sites/default/files/2020-05/aqi-technical-assistance-document-sept2018.pdf

This notebook supports AQI calculations for the following pollutants: O$_3$ (ppm) 8-hour, O$_3$ (ppm) 1-hour, PM$_{2.5}$ ($\mu$g/m$^3$) 24-hour, PM$_{10}$ ($\mu$g/m$^3$) 24-hour, CO (ppm) 8-hour, SO$_2$ (ppb) 1-hour, and NO$_2$ (ppb) 1-hour. Note that only the highest concentration among all of the monitors within a given reporting area should be recorded for each pollutant. 

All data should be input as a dictionary with keys as the pollutant (type string) and the values as the concentration (type float). Note that the only accepted keys are the following (corresponding to the above listed pollutants): 'O3_8hr_ppm', 'O3_1hr_ppm', 'PM2.5_24hr_μg/m^3', 'PM10_24hr_μg/m^3', 'CO_8hr_ppm', 'SO2_1hr_ppb', 'NO2_1hr_ppb'.

### Step 1: Truncate each type of pollutant as follows:
Ozone (ppm) - truncate to 3 decimal places

PM$_{2.5}$ ($\mu$g/m$^3$) - truncate to 1 decimal place

PM$_{10}$ ($\mu$g/m$^3$) - truncate to integer

CO (ppm) - truncate to 1 decimal place

SO$_2$ (ppb) - truncate to integer

NO$_2$ (ppb) - truncate to integer

In [336]:
def truncate_to_three_decimals(number):
    return math.floor(number * 1000) / 1000.0
def truncate_to_one_decimal(number):
    return math.floor(number * 10) / 10.0

In [337]:
def truncation(pollutants,concentrations):
    for i in range(len(pollutants)):
        #ozone case (truncate to 3 decimal places)
        if (pollutants[i] == 'O3_8hr_ppm') or (pollutants[i] == 'O3_1hr_ppm'):
            concentrations[i] = truncate_to_three_decimals(concentrations[i])
        #truncate to 1 decimal place case
        elif (pollutants[i] == 'PM2.5_24hr_μg/m^3') or (pollutants[i] == 'CO_8hr_ppm'):
            concentrations[i] = truncate_to_one_decimal(concentrations[i])
        #truncate to integer
        elif (pollutants[i] == 'PM10_24hr_μg/m^3') or (pollutants[i] == 'SO2_1hr_ppb') or (pollutants[i] == 'NO2_1hr_ppb'):
            concentrations[i] = int(concentrations[i])
        #warn user that one of the keys is not one of the accepted ones
        else:
            print("One of the pollutants entered does not have one of the accepted names! Please ensure that all pollutants are labelled as one of the following: 'O3_8hr_ppm', 'O3_1hr_ppm', 'PM2.5_24hr_μg/m^3', 'PM10_24hr_μg/m^3', 'CO_8hr_ppm', 'SO2_1hr_ppb', 'NO2_1hr_ppb'.")
    return concentrations

### Step 2: Using the table below, find the two breakpoints that contain the concentration and the corresponding AQI range

![Alt Text](/Users/lill771/Downloads/Breakpoints.png)

In [338]:
#Dictionary containing breakpoints for the AQI
BP_AQI_ranges = {
    'O3_8hr_ppm': [
        ((0.000, 0.054), (0, 50)),
        ((0.055, 0.070), (51, 100)),
        ((0.071, 0.085), (101, 150)),
        ((0.086, 0.105), (151, 200)),
        ((0.106, 0.200), (201, 300))
    ],
    'O3_1hr_ppm': [
        ((0.000, 0.125), (0, 101)),
        ((0.125, 0.164), (101, 150)),
        ((0.165, 0.204), (151, 200)),
        ((0.205, 0.404), (201, 300)),
        ((0.405, 0.504), (301, 400)),
        ((0.505, 0.604), (401, 500))
    ],
    'PM2.5_24hr_μg/m^3': [
        ((0.0, 12.0), (0, 50)),
        ((12.1, 35.4), (51, 100)),
        ((35.5, 55.4), (101, 150)),
        ((55.5, 150.4), (151, 200)),
        ((150.5, 250.4), (201, 300)),
        ((250.5, 350.4), (301, 400)),
        ((350.5, 500.4), (401, 500))
    ],
    'PM10_24hr_μg/m^3': [
        ((0, 54), (0, 50)),
        ((55, 154), (51, 100)),
        ((155, 254), (101, 150)),
        ((255, 354), (151, 200)),
        ((355, 424), (201, 300)),
        ((425, 504), (301, 400)),
        ((505, 604), (401, 500))
    ],
    'CO_8hr_ppm': [
        ((0.0, 4.4), (0, 50)),
        ((4.5, 9.4), (51, 100)),
        ((9.5, 12.4), (101, 150)),
        ((12.5, 15.4), (151, 200)),
        ((15.5, 30.4), (201, 300)),
        ((30.5, 40.4), (301, 400)),
        ((40.5, 50.4), (401, 500))
    ],
    'SO2_1hr_ppb': [
        ((0, 35), (0, 50)),
        ((36, 75), (51, 100)),
        ((76, 185), (101, 150)),
        ((186, 304), (151, 200)),
        ((305, 604), (201, 300)),
        ((605, 804), (301, 400)),
        ((805, 1004), (401, 500))
    ],
    'NO2_1hr_ppb': [
        ((0, 53), (0, 50)),
        ((54, 100), (51, 100)),
        ((101, 360), (101, 150)),
        ((361, 649), (151, 200)),
        ((650, 1249), (201, 300)),
        ((1250, 1649), (301, 400)),
        ((1650, 2049), (401, 500)),
        ((2050,10000), (500,500))
    ]
}

In [339]:
#Given the concentration and the given pollutant, return the breakpoints and corresponding AQI range
def find_ranges(concentration, pollutant, BP_AQI_ranges):
    if pollutant in BP_AQI_ranges:
        pollutant_ranges = BP_AQI_ranges[pollutant]
        
        for (BP_lower, BP_upper), (AQI_lower, AQI_upper) in pollutant_ranges:
            if BP_lower <= concentration <= BP_upper:
                return (BP_lower, BP_upper, AQI_lower, AQI_upper)
    
    return None, None, None, None  #If number doesn't fall into any range for the specified category

### Step 3: Calculate the AQI using the following equation: 
$$I_p = \frac{I_{Hi}-I_{Lo}}{BP_{Hi}-BP_{Lo}}(C_p-BP_{Lo})+I_{Lo},$$
where

$I_p$ is the AQI for pollutant $p$

$C_p$ is the truncated concentration of pollutant $p$

$BP_{Hi}$ is the concentration breakpoint that is $\geq C_p$

$BP_{Lo}$ is the concentration breakpoint that is $\leq C_p$

$I_{Hi}$ is the AQI value corresponding to $BP_{Hi}$

$I_{Lo}$ is the AQI value corresponding to $BP_{Lo}.$

### Step 4: Round the AQI to the nearest integer

In [340]:
def AQI(data):
    pollutants = list(data.keys())
    concentrations = list(data.values())
    Ip_vals = np.zeros(len(pollutants))
    #step 1: truncate based on pollutant type
    concentrations = truncation(pollutants,concentrations)
    for i in range(len(pollutants)):
        Cp = concentrations[i]
        if Cp < 0.0:
            Cp = 0.0
        #step 2: find the two breakpoints that contain the concentration
        BP_Lo,BP_Hi,I_Lo,I_Hi = find_ranges(Cp,pollutants[i],BP_AQI_ranges)
        if BP_Lo == None:
            print("Given concentration value does not fall within range of known values corresponding to breakpoints.")
        #step 3: calculate the index
        Ip = (((I_Hi - I_Lo) / (BP_Hi - BP_Lo)) * (Cp - BP_Lo)) + I_Lo
        Ip_vals[i] = Ip
    #step 4: round the index to the nearest integer
    AQI_vals = list(map(round, Ip_vals))
    #the AQI is the highest value calculated amongst all pollutants
    AQI = max(AQI_vals)
    AQI_index = AQI_vals.index(AQI)
    print("The AQI is ", AQI, ", with ", pollutants[AQI_index], " as the responsible pollutant.") 
    return AQI

# Step 10: Using DAC pollutant concentration data to compute AQI

In [341]:
#Compute adjusted values (that incorporate the background pollutants), pass to AQI function, and add AQI column
aqi_values = []
for index, row in Tulare_summed_df.iterrows():
    data_dict = {
        'O3_8hr_ppm': 0.0,
        'O3_1hr_ppm': 0.0,
        'PM2.5_24hr_μg/m^3': max(row['PM2.5 (micrograms/m^3)']) + background_levels['PM2.5 (micrograms/m^3)'],
        'PM10_24hr_μg/m^3': max(row['PM10 (micrograms/m^3)']) + background_levels['PM10 (micrograms/m^3)'],
        'CO_8hr_ppm': max(row['CO (ppm)']) + background_levels['CO (ppm)'],
        'SO2_1hr_ppb': max(row['SO2 (ppb)']) + background_levels['SO2 (ppb)'],
        'NO2_1hr_ppb': max(row['NO2 (ppb)']) + background_levels['NO2 (ppb)'],
    }
    # Pass dictionary to AQI function
    aqi_value = AQI(data_dict)
    aqi_values.append(aqi_value)

The AQI is  125 , with  NO2_1hr_ppb  as the responsible pollutant.
The AQI is  41 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  41 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  40 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  40 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  42 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  42 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  41 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  44 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  41 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  41 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  42 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  41 , with  PM2.5_24hr_μg/m^3  as the responsible pollutant.
The AQI is  40 , with  PM2.5_24hr_μg/m^3  as the responsible pollutan

Source for AQI levels of concern: https://www.airnow.gov/aqi/aqi-basics/

In [342]:
#Add AQI column to the dataframe
Tulare_summed_df['AQI'] = aqi_values

# Output updated dataframe
print(Tulare_summed_df)

                    real_time  \
0   2032-01-01 00:00:00+00:00   
1   2032-01-01 12:00:00+00:00   
2   2032-01-02 12:00:00+00:00   
3   2032-01-03 12:00:00+00:00   
4   2032-01-04 12:00:00+00:00   
..                        ...   
273 2032-09-28 12:00:00+00:00   
274 2032-09-29 12:00:00+00:00   
275 2032-09-30 12:00:00+00:00   
276 2032-10-01 12:00:00+00:00   
277 2032-10-02 12:00:00+00:00   

                                       CO (g/s)_result  \
0    [18.542279322883093, 16.964808966423334, 43.51...   
1    [1.2954760646187116, 1.1823478272488637, 1.475...   
2    [0.09546693670907769, 0.6734632018418635, 0.25...   
3    [0.1657721168752294, 0.16897459625223757, 0.16...   
4    [0.5260600472007679, 0.46606538085640414, 0.58...   
..                                                 ...   
273  [2.116051683505171, 1.8585650096494903, 1.7171...   
274  [0.3946424215757568, 0.4034627229667278, 0.393...   
275  [0.6590222496647592, 1.1999994655346151, 0.812...   
276  [0.392773945806564

In [343]:
#Save the dataframe to a CSV file
Tulare_summed_df.to_csv('Tulare_AQI_MTDC.csv', index=False)